<a href="https://colab.research.google.com/github/JC9427/JC-programming/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **功能**
- 試算表格式整理
- 回看歷史資料(eg：相同分數幾次)
- 清除鍵
- 日期修改方式
- 一鍵複製AI摘要

Googlesheet：https://docs.google.com/spreadsheets/d/1IvgBEfbQ7hWF1xYT26xdbKxP3GhlmDsH2liGxv2SjqI/edit?usp=sharing


In [ ]:
#安裝 gspread
!pip install -q gspread
#安裝 / 升級 Gradio
!pip install -U gradio

In [ ]:
import gradio as gr
import pandas as pd
import gspread

from google.colab import auth
from google.auth import default
from datetime import datetime, timedelta

In [ ]:
# Google Sheet 設定，讓你的程式可以連到 Google 試算表，並準備好欄位格式
SHEET_URL = "https://docs.google.com/spreadsheets/d/1IvgBEfbQ7hWF1xYT26xdbKxP3GhlmDsH2liGxv2SjqI/edit?usp=sharing"
WORKSHEET_NAME = "工作表4"
REQUIRED_COLUMNS = ["日期", "科目", "成績", "科目評論", "總結"]

_gc = None
_ws = None


def setup_gspread(sheet_url, worksheet_name):
    global _gc, _ws

    if _gc is None or _ws is None:
        print("─── 正在進行 Google Sheet 身分驗證和連線... ───")
        try:
            auth.authenticate_user()
            creds, _ = default()
            _gc = gspread.authorize(creds)
            sh = _gc.open_by_url(sheet_url)
            _ws = sh.worksheet(worksheet_name)
            print("─── Google Sheet 連線成功。───")

            current_header = _ws.row_values(1)
            if len(current_header) == 0:
                _ws.append_row(REQUIRED_COLUMNS)
            elif current_header != REQUIRED_COLUMNS:
                print("目前欄位名稱和程式設定不同，請確認工作表第一列是否為：", REQUIRED_COLUMNS)

        except Exception as e:
            print(f"Google Sheet 連線失敗: {e}")
            _gc = None
            _ws = None

In [ ]:
# 決定今天日期（台灣時間），決定資料要插進試算表的哪一列
def get_today_date():
    """
    回傳台灣時間今天日期，格式：YYYY-MM-DD
    """
    taiwan_time = datetime.utcnow() + timedelta(hours=8)
    return taiwan_time.strftime("%Y-%m-%d")


def normalize_selected_date(selected_date):
    """
    把 Gradio 傳進來的日期整理成 YYYY-MM-DD
    """
    if selected_date is None or selected_date == "":
        return get_today_date()

    if isinstance(selected_date, datetime):
        return selected_date.strftime("%Y-%m-%d")

    if isinstance(selected_date, (int, float)):
        dt = datetime.utcfromtimestamp(selected_date) + timedelta(hours=8)
        return dt.strftime("%Y-%m-%d")

    text = str(selected_date).strip()

    if len(text) >= 10 and text[4] == "-" and text[7] == "-":
        return text[:10]

    return get_today_date()


def find_insert_row_by_date(target_date):
    """
    根據日期決定插入位置
    讓補登資料可以排到對應日期區塊
    """
    global _ws

    all_values = _ws.get_all_values()

    if len(all_values) <= 1:
        return 2

    target_dt = datetime.strptime(target_date, "%Y-%m-%d")

    for row_index in range(2, len(all_values) + 1):
        row_data = all_values[row_index - 1]

        if len(row_data) == 0:
            return row_index

        row_date = str(row_data[0]).strip()

        if row_date == "":
            return row_index

        try:
            row_dt = datetime.strptime(row_date[:10], "%Y-%m-%d")
        except:
            continue

        if row_dt > target_dt:
            return row_index

    return len(all_values) + 1

In [ ]:
def get_existing_records():
    """
    取得 Google Sheet 目前所有資料（不含標題列）
    """
    global _ws
    try:
        return _ws.get_all_records()
    except Exception as e:
        print(f"讀取既有資料失敗: {e}")
        return []


def get_subject_history(records, subject):
    """
    找出某科過去所有成績
    """
    history = []

    for row in records:
        row_subject = str(row.get("科目", "")).strip()
        row_grade = row.get("成績", "")

        if row_subject == subject:
            try:
                history.append(int(row_grade))
            except:
                pass

    return history


def build_subject_comment(subject, current_grade, history):
    """
    根據同科歷史成績，產生科目評論
    """
    if len(history) == 0:
        return f"{subject}本次成績為{current_grade}分，這是第一次紀錄。"

    last_grade = history[-1]
    same_count = history.count(current_grade)

    if current_grade > last_grade:
        diff = current_grade - last_grade
        return f"{subject}本次成績為{current_grade}分，和上一次相比進步{diff}分。"
    elif current_grade < last_grade:
        diff = last_grade - current_grade
        return f"{subject}本次成績為{current_grade}分，和上一次相比退步{diff}分。"
    else:
        total_same_times = same_count + 1
        return f"{subject}本次成績為{current_grade}分，這是第{total_same_times}次拿到相同成績。"


def build_subject_advice(subject, grade):
    """
    根據科目與分數，產生各科建議或鼓勵文字
    """
    subject = str(subject).strip()

    if grade >= 90:
        if subject == "國文":
            return "國文表現很好，請繼續保持目前的閱讀與表達能力。"
        elif subject == "英文":
            return "英文表現很好，請繼續保持目前的學習狀態。"
        elif subject == "數學":
            return "數學表現很好，請繼續保持目前的計算與理解能力。"
        elif subject == "自然":
            return "自然表現很好，請繼續保持對觀念與應用的掌握。"
        elif subject == "社會":
            return "社會表現很好，請繼續保持目前的理解與記憶表現。"
        else:
            return f"{subject}表現很好，請繼續保持目前的學習狀態。"

    elif grade >= 80:
        if subject == "國文":
            return "國文表現不錯，可多加強閱讀理解與作文表達。"
        elif subject == "英文":
            return "英文表現不錯，可持續練習單字、文法與閱讀速度。"
        elif subject == "數學":
            return "數學表現不錯，可多練習題型變化與計算熟練度。"
        elif subject == "自然":
            return "自然表現不錯，可再加強觀念整理與題目應用。"
        elif subject == "社會":
            return "社會表現不錯，可再加強重點整理與記憶熟練度。"
        else:
            return f"{subject}表現不錯，可持續練習，讓成績更加穩定。"

    elif grade >= 60:
        if subject == "國文":
            return "國文可多加強閱讀理解、字詞運用與作文練習。"
        elif subject == "英文":
            return "英文可多背單字，並加強文法與句型練習。"
        elif subject == "數學":
            return "數學可從基礎觀念開始複習，並多做練習題。"
        elif subject == "自然":
            return "自然可多複習基本觀念，並加強題目應用練習。"
        elif subject == "社會":
            return "社會可多整理重點，透過反覆複習加強記憶。"
        else:
            return f"{subject}可多複習基礎內容，並透過練習加強理解。"

    else:
        if subject == "國文":
            return "國文需要加強基礎閱讀與字詞理解，建議從課文重點開始複習。"
        elif subject == "英文":
            return "英文需要加強基礎單字與文法，建議先從基本句型開始練習。"
        elif subject == "數學":
            return "數學需要加強基礎觀念與計算能力，建議先複習基本題型。"
        elif subject == "自然":
            return "自然需要加強基礎觀念，建議先從課本重點與基本題開始複習。"
        elif subject == "社會":
            return "社會需要加強基礎記憶與理解，建議先整理課本重點再複習。"
        else:
            return f"{subject}需要先加強基礎觀念，建議從課本內容與基本題開始複習。"


def build_overall_summary(clean_grades, history_map):
    """
    生成整體總結 + 各科建議
    """
    improved = 0
    declined = 0
    same = 0
    first_time = 0

    for _, subject, grade in clean_grades:
        history = history_map.get(subject, [])

        if len(history) == 0:
            first_time += 1
        else:
            last_grade = history[-1]
            if grade > last_grade:
                improved += 1
            elif grade < last_grade:
                declined += 1
            else:
                same += 1

    subject_count = len(clean_grades)
    summary_parts = [f"本次共輸入{subject_count}科成績。"]

    if improved > 0:
        summary_parts.append(f"其中有{improved}科比上次進步。")
    if declined > 0:
        summary_parts.append(f"有{declined}科比上次退步。")
    if same > 0:
        summary_parts.append(f"有{same}科和上次相同。")
    if first_time > 0:
        summary_parts.append(f"有{first_time}科是第一次紀錄。")

    advice_parts = []
    for _, subject, grade in clean_grades:
        advice_parts.append(f"{subject}：{build_subject_advice(subject, grade)}")

    summary_text = "".join(summary_parts)
    advice_text = "\n".join(advice_parts)

    return summary_text + "\n\n【各科建議】\n" + advice_text

In [ ]:
# 讀取以前的資料，給予評價
def process_grades_and_summary(selected_date, grade_data):
    """
    處理 Gradio 介面輸入的成績資料，寫入 Google Sheet 並生成摘要。
    """
    global _gc, _ws

    try:
        if _ws is None:
            setup_gspread(SHEET_URL, WORKSHEET_NAME)
            if _ws is None:
                return "Google Sheet 未能成功連線，請檢查錯誤訊息並重試。", "摘要生成失敗。"

        if not grade_data:
            return "沒有輸入任何成績，請輸入科目和成績。", ""

        current_date = normalize_selected_date(selected_date)
        clean_grades = []

        for row in grade_data:
            if len(row) < 2:
                continue

            subject = str(row[0]).strip()
            grade_str = str(row[1]).strip()

            if subject == "" and grade_str == "":
                continue

            if subject == "":
                return "科目不能空白。", ""

            try:
                grade = int(grade_str)
            except ValueError:
                return f"科目「{subject}」的成績「{grade_str}」無效，成績必須是數字。", ""

            clean_grades.append([current_date, subject, grade])

        if not clean_grades:
            return "沒有有效的成績資料。", ""

        existing_records = get_existing_records()

        subject_comments = {}
        history_map = {}

        for _, subject, grade in clean_grades:
            history = get_subject_history(existing_records, subject)
            history_map[subject] = history
            subject_comments[subject] = build_subject_comment(subject, grade, history)

        overall_summary = build_overall_summary(clean_grades, history_map)

        new_rows_for_sheet = []
        summary_for_output = "【各科評論】\n"

        for i, record in enumerate(clean_grades):
            date, subject, grade = record
            subject_comment = subject_comments[subject]
            total_summary = overall_summary if i == 0 else ""

            new_rows_for_sheet.append([
                date,
                subject,
                grade,
                subject_comment,
                total_summary
            ])

            summary_for_output += f"{subject}：{subject_comment}\n"

        summary_for_output += f"\n【整體總結】\n{overall_summary}"

        insert_row = find_insert_row_by_date(current_date)
        _ws.insert_rows(new_rows_for_sheet, row=insert_row)

        try:
            _ws.format(
                "A:A",
                {
                    "numberFormat": {
                        "type": "DATE",
                        "pattern": "yyyy-mm-dd"
                    }
                }
            )
        except Exception as e:
            print(f"設定日期格式失敗: {e}")

        sheet_message = f"成績已成功寫入 Google Sheet，日期為 {current_date}。"
        return sheet_message, summary_for_output

    except Exception as e:
        print(f"process_grades_and_summary 發生錯誤：{e}")
        return f"處理失敗：{e}", "摘要生成失敗，請查看 Colab 錯誤訊息。"

In [ ]:
#確認有沒有成功連到試算表
setup_gspread(SHEET_URL, WORKSHEET_NAME)

if _ws is None:
    print("注意：Google Sheet 工作表物件 (_ws) 仍為 None，表示連線可能有問題。")
else:
    print("Google Sheet 工作表物件 (_ws) 已成功初始化，連線已建立。")

─── 正在進行 Google Sheet 身分驗證和連線... ───
─── Google Sheet 連線成功。───
Google Sheet 工作表物件 (_ws) 已成功初始化，連線已建立。


In [ ]:
def clear_grades():
    return get_today_date(), [["", ""]], "", ""

In [ ]:
# 建立一個可以輸入成績 → 分析 → 存進試算表 → 顯示結果 → 複製內容的完整網頁
with gr.Blocks() as demo:
    gr.Markdown("# 成績輸入與 AI 摘要工具")
    gr.Markdown("請先確認日期（預設今天，可手動修改補登舊資料），再在下方表格中輸入學生的科目和成績，然後點擊「送出」。系統會將資料寫入 Google Sheet 並生成 AI 摘要。")

    with gr.Row():
        with gr.Column():
            input_date = gr.DateTime(
                label="選擇日期（可手動修改或用日曆選）",
                value=get_today_date(),
                include_time=False,
                type="string"
            )

            grade_input = gr.Dataframe(
                headers=["科目", "成績"],
                value=[["", ""]],
                type="array",
                row_count=1,
                col_count=(2, "fixed"),
                label="輸入科目與成績（點擊最後一行可新增）"
            )

            with gr.Row():
                submit_button = gr.Button("送出")
                clear_button = gr.Button("清除成績")

        with gr.Column():
            sheet_output = gr.Textbox(label="Google Sheet 處理狀態")
            summary_output = gr.Textbox(
                label="AI 摘要",
                lines=15
            )
            copy_button = gr.Button("複製 AI 摘要")

    submit_button.click(
        process_grades_and_summary,
        inputs=[input_date, grade_input],
        outputs=[sheet_output, summary_output]
    )

    clear_button.click(
        clear_grades,
        inputs=None,
        outputs=[input_date, grade_input, sheet_output, summary_output]
    )

    copy_button.click(
        fn=None,
        inputs=[summary_output],
        outputs=[],
        js="(text) => { navigator.clipboard.writeText(text || ''); }"
    )

demo.launch()

/tmp/ipykernel_45105/3532386760.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  taiwan_time = datetime.utcnow() + timedelta(hours=8)
/usr/local/lib/python3.12/dist-packages/gradio/components/dataframe.py:192: UserWarning: The `col_count` parameter is deprecated and will be removed. Please use `column_count` instead.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://93e999dab418f879c1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
